# Chapter 10: Tool Engineering
## Topic 7: Testing Tool-Calling Reliability

**One notebook. Theory + Code together.**


## Part A: Theory

### 1. Concept, Intuition, and Why This Exists

- Every prior topic in this chapter measured *something* about tool-calling behavior as part of demonstrating its point — Topic 4 measured schema-wording effects on triggering accuracy (86% vs 100%), Topic 6 measured multi-tool selection accuracy across single-tool, no-tool, and both-tools-needed cases. This closing topic makes that measurement discipline explicit and systematic: what does a genuine, ongoing *test suite* for tool-calling reliability actually look like, rather than a one-off measurement made once to prove a specific point?
- The core reframe this topic makes, extending a pattern already established repeatedly across this notebook series (Chapter 7 Topic 9's retrieval evaluation, Chapter 9 Topic 7's hallucination-rate measurement): tool-calling reliability is not a single pass/fail property verified once — it's a set of distinct, measurable dimensions that should be tracked as a project's tools, schemas, and prompts evolve over time, using the same evidence-based discipline this entire notebook series has repeatedly applied to every other component.
- Three genuinely distinct dimensions this topic identifies, each requiring its own kind of test, building directly on this chapter's own prior topics: **does the model call the right tool at the right time** (Topic 1's triggering and Topic 6's selection, now formalized into a repeatable test), **does the model call it with correct, well-formed arguments** (directly connecting to Topic 4's schema-quality discussion), and **does the system handle the tool's result — success, legitimate negative, or genuine error — correctly afterward** (Topic 5's error-handling categories, now needing their own dedicated tests too).


### 2. Internal Working — Step by Step

**Building a genuine test suite for tool-calling reliability, as three distinct test categories:**

1. **Triggering/selection accuracy tests** — a labeled set of realistic inputs, each paired with the correct tool (or correct combination of tools, or correctly *no* tool) that should be selected, exactly as demonstrated in this chapter's own Topics 1, 4, and 6. The test simply runs each labeled input through the actual agent and checks whether the tools actually selected match the labeled correct answer — this is a pass/fail check per test case, aggregated into an accuracy percentage exactly like every other accuracy metric in this notebook series.
2. **Argument-correctness tests** — a separate labeled set specifically checking, for cases where a tool *should* be called, whether the arguments the model actually constructs are correct and well-formed — not just "was the right tool selected" but "was it called with the right input." This directly tests what Topic 4's schema-quality principles and Topic 1's dispatch-validation were designed to influence and catch, respectively — a test here reveals whether those upstream design choices are actually working as intended, on real cases, not just in the specific illustrative example each topic used to make its point.
3. **Result-handling tests** — this is the category most distinct from ordinary accuracy testing, and needs genuinely different test infrastructure: rather than checking whether the model behaves correctly given normal conditions, these tests deliberately *inject* each of Topic 5's failure categories (a controlled, simulated transient error; a controlled permanent error; a genuine legitimate negative result) and verify that the surrounding system handles each one according to Topic 5's principles — retrying only the transient case, failing fast on the permanent case, and producing the correct distinct result shape for each, regardless of what the model itself does with that result afterward.
4. **Regression testing across changes** — exactly as Topic 4 argued for schema wording changes specifically, and as Chapter 7 Topic 9 established for retrieval configuration changes generally, every one of these three test categories should be re-run whenever anything upstream changes — a new tool added (Topic 6), a schema rewritten (Topic 4), a prompt updated (Chapter 9 Topic 6) — to confirm the change didn't silently regress a capability that was previously working correctly.


### 3. How This Is Implemented in This Project

- The labeled test sets already built and used illustratively in this chapter's own Topics 1, 4, and 6 (single-tool triggering accuracy, schema-wording comparison, multi-tool selection accuracy) are not just one-off demonstrations — they're the actual starting material for this project's genuine, ongoing tool-calling test suite. The pattern established repeatedly (a list of labeled inputs, a function that runs the real logic being tested, an aggregated accuracy calculation) is directly reusable as standing test infrastructure, not something that needs to be built from scratch for this topic.
- Result-handling tests specifically need the controllable, simulated failure infrastructure built in Topic 5 (`SimulatedFlakyDependency`, deliberately configurable to fail a specific number of times or permanently) — this is exactly the kind of test double needed to reliably and repeatably exercise the transient-error and permanent-error code paths, since real production failures are, by nature, not reliably reproducible on demand for testing purposes.
- This project's existing `run_agent()` function itself is the natural target these tests should run against end-to-end wherever feasible — testing the actual agent loop's behavior (given a real or simulated model response) rather than only testing isolated pieces of logic in isolation, since the goal is confidence in the system's actual, integrated behavior, not just its individual components considered separately.


### 4. Real-World Issues, Edge Cases, Debugging, Monitoring, Scaling, Latency, Cost, Security, Deployment

- **Testing real model tool-calling behavior costs real API calls, unlike testing pure application logic** — every test case that actually invokes a live model to check its tool-selection behavior costs a real request; a large, comprehensive test suite run frequently (say, on every code change) can accumulate meaningful cost, which is exactly why Topic 5's simulated dependencies and this topic's own emphasis on isolating what's actually being tested (application logic vs model behavior) matters practically, not just for test-writing convenience.
- **A test suite that only covers "happy path" triggering cases misses exactly the failure modes this notebook series has repeatedly found matter most** — Chapter 9 Topic 7's own measured finding (a 3.3x higher hallucination rate specifically for Hinglish-heavy content) is a direct instance of a broader principle: a good test suite deliberately includes edge cases and known-difficult segments, not just the easy, obviously-correct cases that would pass regardless of whether the underlying system is actually reliable.
- **Regression testing requires stable baselines to compare against** — without a saved record of a previous accuracy measurement, a new measurement has nothing to be compared to, and "did this change help or hurt" becomes an unanswerable question rather than a measured one. Storing test results over time (not just running tests and discarding the numbers) is what makes genuine before-and-after comparison possible at all.
- **Debugging a newly-failing test case requires the same layered attribution discipline established throughout this notebook series** (Chapter 8 Topic 3's faithfulness/relevance/correctness framework, applied here to tool-calling specifically) — a triggering-accuracy regression could stem from a schema change (Topic 4), a system-prompt change, or a genuinely new, harder query pattern the test set didn't previously include — conflating these wastes debugging effort exactly as every other "collapsed failure category" mistake in this notebook series has warned against.
- **Monitoring in production is a complement to, not a replacement for, this kind of pre-deployment testing** — Chapter 9 Topic 7's ongoing hallucination-rate monitoring and this topic's test suite serve related but distinct purposes: tests catch known, anticipated failure patterns before deployment using controlled, labeled cases; production monitoring catches genuinely novel failure patterns that emerge from real, unanticipated traffic after deployment. Both are necessary; neither substitutes for the other.


### 5. Design Decisions, Trade-offs, and Real-Time Dilemmas

- **Testing against a real model call vs a simulated one, for each specific test category:** triggering and selection accuracy tests ideally test real model behavior at least periodically, since that's genuinely what's being evaluated — but running every test against a live model call on every code change is expensive and slow. Result-handling tests, by contrast, are testing the *application's* logic (retry behavior, error-shape construction) rather than the model's behavior at all, and should almost always use Topic 5's simulated, controllable failure infrastructure rather than real API calls, since what's being verified there has nothing to do with model behavior specifically.
- **How large a labeled test set needs to be to produce a trustworthy accuracy number:** a very small test set (like this chapter's own illustrative 4-9 case examples) is enough to demonstrate a principle clearly, but produces a noisy, easily-swung accuracy percentage — a single additional case can move the measured accuracy by more than 10 percentage points at that scale. A genuinely trustworthy, production-grade test suite needs a meaningfully larger, more representative set, mirroring exactly the same size-and-representativeness concern Chapter 7 Topic 9 raised for retrieval evaluation sets generally.
- **How often to re-run the full test suite — on every code change, on a schedule, or only before major releases:** more frequent testing catches regressions faster but costs more (especially for test categories requiring real model calls), while less frequent testing risks a regression going undetected for longer before it's caught — this should be tuned based on how frequently the tested components (schemas, prompts, tool sets) actually change in practice, not fixed arbitrarily.


### 6. Alternatives and When to Use Each

- **Ad hoc, illustrative testing (this chapter's own Topics 1, 4, 6 as originally presented):** valuable for demonstrating and validating a specific principle clearly during development, but insufficient as ongoing production infrastructure on its own — the small, illustrative test sets used to make a point are a starting point, not a finished test suite.
- **A genuine, larger, regularly-run test suite covering triggering/selection, argument-correctness, and result-handling as distinct categories (this topic's recommended approach):** the right standard for a production system, extending the illustrative examples from earlier topics into real, ongoing, evidence-generating infrastructure.
- **Relying solely on production monitoring (Chapter 9 Topic 7's approach) with no pre-deployment testing at all:** catches real problems, but only after they've already affected real customers — pre-deployment testing exists specifically to catch known, anticipated failure patterns *before* they reach production traffic, a genuinely different and complementary purpose.


### 7. Common Mistakes and Production Failures

- Treating a small, illustrative test set (built to demonstrate one specific point clearly) as sufficient, ongoing test coverage, without expanding it into a larger, more representative, regularly-maintained suite.
- Testing only "happy path" cases that would pass regardless of whether the underlying system is genuinely reliable, missing exactly the edge cases and known-difficult segments (like Hinglish-heavy content, per Chapter 9 Topic 7) that matter most.
- Running expensive, real-model-call tests for result-handling scenarios (retry logic, error-shape construction) that should instead use Topic 5's cheap, controllable simulated failure infrastructure, since those tests are actually about application logic, not model behavior.
- Not saving test results over time, making genuine before-and-after regression comparison impossible when a schema, prompt, or tool set changes.
- Conflating a triggering-accuracy regression's root cause (a schema change, a prompt change, or a genuinely new query pattern) without applying the same layered attribution discipline used throughout this notebook series for other failure categories.


### 8. Lead-Level Interview Questions

**Basic**

- Q: What are the three distinct dimensions of tool-calling reliability this topic identifies, and why can't one test cover all three?
  A: Whether the right tool (or combination of tools) gets selected at all, whether it gets called with correct, well-formed arguments, and whether the system correctly handles whatever result comes back (success, legitimate negative, or genuine error). These are genuinely different failure modes — a tool can be correctly selected but called with malformed arguments, or correctly selected and called correctly but have its error result mishandled downstream — so each needs its own dedicated test category to catch failures specific to that dimension.

- Q: Why should result-handling tests (retry logic, error handling) generally use simulated failures rather than real model calls?
  A: These tests are verifying the application's own logic — does it retry a transient error correctly, does it fail fast on a permanent error, does it construct the correct distinct result shape for each case — none of which has anything to do with model behavior specifically. Using Topic 5's controllable simulated dependencies makes these failure scenarios reliably reproducible on demand, which real production failures, by nature, are not.

**Intermediate**

- Q: This chapter's own Topics 1, 4, and 6 each used a small labeled test set (often fewer than 10 cases) to illustrate a specific measured point. Why is this insufficient as an ongoing production test suite on its own?
  A: A test set that small produces a noisy accuracy measurement, where a single additional or different case can shift the reported accuracy by a large margin — useful for clearly demonstrating a specific principle, but not for producing a trustworthy, stable reliability number to track over time. A genuinely production-grade test suite needs a larger, more representative set of cases, mirroring the same size-and-representativeness concern already raised for retrieval evaluation sets (Chapter 7 Topic 9).

- Q: How does this topic's test suite relate to Chapter 9 Topic 7's ongoing production hallucination-rate monitoring?
  A: They serve complementary, not overlapping, purposes. This topic's tests run against known, anticipated, labeled cases before deployment, catching regressions in tool-calling behavior before real customers are affected. Chapter 9 Topic 7's monitoring observes real, live production traffic after deployment, catching genuinely novel failure patterns that a pre-deployment test set, by definition, couldn't have anticipated. Both are necessary parts of a complete reliability strategy.

**Advanced**

- Q: Design a regression-testing process for tool-calling reliability that would catch a schema wording change (Topic 4) that accidentally reduces triggering accuracy.
  A: Maintain a saved baseline of the current triggering/selection accuracy on a representative, sufficiently large labeled test set. Before deploying any schema change, re-run this exact same test set against the proposed new schema, and compare the new accuracy against the saved baseline. A measured drop, even a modest one, should block the change until investigated — exactly the same evidence-based validation discipline Topic 4 itself argued for schema changes, now formalized as a required, repeatable step rather than an ad hoc, case-by-case check.

- Q: A project's tool-calling test suite passes consistently, but production monitoring later reveals a real reliability problem the tests never caught. How would you investigate whether this represents a genuine testing gap versus an inherent limitation of pre-deployment testing?
  A: First check whether the production failure pattern involves a query type, segment, or edge case that simply isn't represented in the existing labeled test set at all — if so, this is a genuine, fixable testing gap, and the fix is expanding the test set to include this newly-discovered pattern, exactly mirroring how Chapter 9 Topic 7's segmented monitoring surfaced the Hinglish-specific hallucination-rate finding that then presumably should inform which segments future test sets deliberately include. If the failure pattern is something a labeled test set genuinely couldn't have anticipated (a truly novel, previously-unseen scenario), this reflects the inherent, expected complementary relationship between pre-deployment testing and production monitoring, not a testing failure — pre-deployment tests can only catch what they're designed to check for, which is exactly why production monitoring remains necessary regardless of how thorough testing becomes.

**Scenario-based**

- Q: You're asked to build a test suite for a newly-added tool (following Topic 6's multi-tool agent pattern) before it goes to production. Walk through what you'd build.
  A: First, a triggering/selection accuracy test: a labeled set of realistic emails, each marked with whether this new tool should be selected (alone or in combination with existing tools), covering clear-yes, clear-no, and ambiguous boundary cases, following exactly the methodology already demonstrated in this chapter's own Topics 1 and 6. Second, an argument-correctness test: cases where the tool should be called, checking whether the constructed arguments are well-formed and correct, connecting to Topic 4's schema-quality principles. Third, a result-handling test: using Topic 5's simulated failure infrastructure, deliberately inject a transient failure, a permanent failure, and a legitimate negative result for this new tool specifically, verifying each produces the correct distinct result shape and handling behavior. Finally, save this initial measurement as a baseline for future regression comparison, exactly as this topic's regression-testing principle requires.

**Follow-up questions to expect:**

- "How would you estimate the right size for a labeled test set to get a trustworthy accuracy number?"
- "What would you do if a test suite consistently shows high accuracy, but real users report frequent tool-calling problems?"


### 9. Hidden Concepts / Prerequisites Worth Knowing

- **This topic's three-category test taxonomy (triggering/selection, argument-correctness, result-handling) is a specific application of a general software testing principle: testing distinct concerns separately rather than relying on one broad, end-to-end test to catch everything.** This mirrors the general software-engineering distinction between unit tests (isolated, specific behavior) and integration tests (the whole system working together), applied here specifically to the tool-calling dimension of an LLM-based system.
- **Using simulated, controllable failures (Topic 5's infrastructure) for result-handling tests is a specific instance of the general "test double" pattern in software testing** — using a controllable stand-in for a real, hard-to-reliably-reproduce dependency, a technique that predates and extends well beyond LLM systems specifically.
- **This topic closes Chapter 10's arc by turning every prior topic's illustrative measurement into standing, reusable infrastructure** — Topic 1's triggering test, Topic 4's schema comparison, Topic 5's simulated failures, and Topic 6's selection-accuracy test were all, in effect, previews of this topic's actual subject, each demonstrating a specific principle using the exact methodology a genuine test suite would use continuously, not just once.

### 10. Quick Revision Summary (for last-minute interview prep)

> Testing tool-calling reliability means tracking three genuinely distinct, measurable dimensions, not treating tool-calling as a single pass/fail property: whether the right tool (or combination) gets selected at all (Topics 1 and 6's triggering/selection tests), whether it's called with correct, well-formed arguments (connecting to Topic 4's schema-quality work), and whether the surrounding system correctly handles whatever result comes back — success, legitimate negative, or genuine error (Topic 5's three failure categories, each needing its own dedicated test). Triggering and selection tests should ideally exercise real model behavior, at least periodically, since that's genuinely what's being measured; result-handling tests should almost always use Topic 5's cheap, controllable simulated failures, since they're actually testing application logic, not model behavior. Every one of this chapter's own illustrative test sets (Topics 1, 4, 6) demonstrates the right methodology at small scale — the genuine next step for production use is expanding these into larger, more representative, regularly re-run test suites with saved baselines, enabling real regression detection whenever a schema, prompt, or tool set changes. This pre-deployment testing discipline complements, but never replaces, Chapter 9 Topic 7's ongoing production monitoring — tests catch known, anticipated patterns before deployment; monitoring catches genuinely novel patterns that emerge from real traffic afterward.


### Module 1: A Real Three-Category Test Suite, Structured as Reusable Infrastructure

Build an actual TestSuite class combining triggering/selection tests, argument-correctness tests, and result-handling tests -- not illustrative one-offs, but structured, reusable, extensible infrastructure.

In [1]:

# ------------------------------------------------------------------
# MODULE 1: A real, structured three-category test suite
# ------------------------------------------------------------------

import re
import json

def validate_fd_reference(reference_number: str) -> dict:
    FD_RECORDS = {"BJ2019FD7717": {"customer_name": "Shobha Chopra", "status": "Closed (Premature)"}}
    record = FD_RECORDS.get(reference_number.strip())
    if record is None:
        return {"reference_number": reference_number, "found": False}
    return {"reference_number": reference_number, "found": True, **record}


def simulate_agent_tool_selection(email_text: str) -> dict:
    """Simulates the agent's full decision: which tool to call (if any)
    and what arguments to use -- standing in for a real run_agent()
    call, so this test suite's STRUCTURE is real and reusable even
    though the underlying model call is simulated here."""
    match = re.search(r'\b[A-Z]{2}\d{4}FD\d{4}\b', email_text)
    if match:
        return {"tool": "validate_fd_reference", "arguments": {"reference_number": match.group(0)}}
    return {"tool": None, "arguments": None}


class ToolCallingTestSuite:
    """A REAL, structured test suite -- three distinct test categories,
    each with its own labeled cases and its own pass/fail logic,
    aggregated into a single reliability report."""

    def __init__(self):
        self.triggering_cases = []
        self.argument_cases = []
        self.result_handling_cases = []

    def add_triggering_case(self, email: str, correct_tool: str):
        self.triggering_cases.append((email, correct_tool))

    def add_argument_case(self, email: str, correct_arguments: dict):
        self.argument_cases.append((email, correct_arguments))

    def run_triggering_tests(self) -> dict:
        correct = 0
        for email, correct_tool in self.triggering_cases:
            decision = simulate_agent_tool_selection(email)
            if decision["tool"] == correct_tool:
                correct += 1
        return {"category": "triggering/selection", "correct": correct, "total": len(self.triggering_cases),
                "accuracy": correct / len(self.triggering_cases) if self.triggering_cases else 0.0}

    def run_argument_tests(self) -> dict:
        correct = 0
        for email, correct_args in self.argument_cases:
            decision = simulate_agent_tool_selection(email)
            if decision["arguments"] == correct_args:
                correct += 1
        return {"category": "argument-correctness", "correct": correct, "total": len(self.argument_cases),
                "accuracy": correct / len(self.argument_cases) if self.argument_cases else 0.0}

    def run_full_suite(self) -> list:
        return [self.run_triggering_tests(), self.run_argument_tests()]


# build a LARGER labeled set than this chapter's earlier illustrative
# examples -- still modest, but structured as real, extensible test data
suite = ToolCallingTestSuite()
triggering_data = [
    ("What is the penalty for my FD BJ2019FD7717?", "validate_fd_reference"),
    ("My loan EMI bounced, please help.", None),
    ("Is my FD BJ2019FD7717 active?", "validate_fd_reference"),
    ("App login is not working.", None),
    ("Reference BJ2019FD7717 needs checking.", "validate_fd_reference"),
    ("Insurance claim rejected, please advise.", None),
]
for email, tool in triggering_data:
    suite.add_triggering_case(email, tool)

argument_data = [
    ("Check my FD BJ2019FD7717 please.", {"reference_number": "BJ2019FD7717"}),
    ("What about reference BJ2019FD7717?", {"reference_number": "BJ2019FD7717"}),
]
for email, args in argument_data:
    suite.add_argument_case(email, args)

print("=" * 70)
print("MODULE 1: STRUCTURED THREE-CATEGORY TEST SUITE")
print("=" * 70)

results = suite.run_full_suite()
for r in results:
    r_category = r["category"]
    r_correct = r["correct"]
    r_total = r["total"]
    r_accuracy = r["accuracy"]
    print(f"\n[{r_category}]")
    print(f"  {r_correct}/{r_total} correct -- accuracy: {r_accuracy*100:.0f}%")

print("\nThis is REUSABLE, STRUCTURED infrastructure -- not a one-off")
print("illustrative measurement. New cases can be added via add_triggering_case()")
print("and add_argument_case() as the project's real query patterns grow.")
print("\nModule 1 complete. Run Module 2 next.")


MODULE 1: STRUCTURED THREE-CATEGORY TEST SUITE

[triggering/selection]
  6/6 correct -- accuracy: 100%

[argument-correctness]
  2/2 correct -- accuracy: 100%

This is REUSABLE, STRUCTURED infrastructure -- not a one-off
illustrative measurement. New cases can be added via add_triggering_case()
and add_argument_case() as the project's real query patterns grow.

Module 1 complete. Run Module 2 next.


### Module 2: Result-Handling Tests Using Topic 5's Simulated Failure Infrastructure

Add the third test category -- deliberately injecting each of Topic 5's three failure categories and verifying correct handling, using cheap, controllable simulation, never real API calls.

In [2]:

# ------------------------------------------------------------------
# MODULE 2: Result-handling tests, using Topic 5's simulated failures
# ------------------------------------------------------------------

import time

class TransientError(Exception):
    pass

class PermanentError(Exception):
    pass


class SimulatedFlakyDependency:
    """Reused from Topic 5 -- the controllable test double that makes
    result-handling tests reliably reproducible on demand."""
    def __init__(self, fail_count: int = 0, permanent_failure: bool = False):
        self.fail_count = fail_count
        self.permanent_failure = permanent_failure
        self.attempts_made = 0

    def query(self, reference_number: str) -> dict:
        self.attempts_made += 1
        if self.permanent_failure:
            raise PermanentError("Malformed query -- will fail every time.")
        if self.attempts_made <= self.fail_count:
            raise TransientError(f"Attempt {self.attempts_made}: timed out.")
        return {"found": True, "reference_number": reference_number}


def call_with_retry(query_func, ref: str, max_attempts: int = 3, base_delay: float = 0.05) -> dict:
    """Same retry-with-backoff logic from Topic 5, reused here as the
    REAL logic under test, not reimplemented for this test."""
    for attempt in range(1, max_attempts + 1):
        try:
            result = query_func(ref)
            return {"status": "success", "result": result, "attempts_used": attempt}
        except TransientError:
            if attempt < max_attempts:
                time.sleep(base_delay * attempt)
        except PermanentError as e:
            return {"status": "permanent_failure", "error": str(e), "attempts_used": attempt}
    return {"status": "retries_exhausted", "attempts_used": max_attempts}


def run_result_handling_tests() -> dict:
    """The THIRD test category: deliberately inject each Topic 5
    failure scenario and verify the system handles each one correctly
    according to Topic 5's principles."""
    test_cases = [
        ("legitimate_negative", SimulatedFlakyDependency(fail_count=0), "success", None),
        ("transient_resolves", SimulatedFlakyDependency(fail_count=1), "success", None),
        ("transient_exhausts", SimulatedFlakyDependency(fail_count=10), "retries_exhausted", None),
        ("permanent_error", SimulatedFlakyDependency(fail_count=0, permanent_failure=True),
         "permanent_failure", 1),  # expect EXACTLY 1 attempt -- no wasted retries
    ]

    correct = 0
    details = []
    for label, dep, expected_status, expected_attempts in test_cases:
        outcome = call_with_retry(dep.query, "BJ2019FD7717", max_attempts=3)
        status_ok = outcome["status"] == expected_status
        attempts_ok = (expected_attempts is None) or (outcome["attempts_used"] == expected_attempts)
        is_correct = status_ok and attempts_ok
        correct += is_correct
        details.append((label, outcome, is_correct))

    return {"category": "result-handling", "correct": correct, "total": len(test_cases),
            "accuracy": correct / len(test_cases), "details": details}


print("=" * 70)
print("MODULE 2: RESULT-HANDLING TESTS (Topic 5's failure categories)")
print("=" * 70)

result_handling_report = run_result_handling_tests()

for label, outcome, is_correct in result_handling_report["details"]:
    status_label = "PASS" if is_correct else "FAIL"
    print(f"\n[{label}] -> {status_label}")
    outcome_status = outcome["status"]
    outcome_attempts = outcome["attempts_used"]
    print(f"  Actual outcome: status={outcome_status}, attempts={outcome_attempts}")

rh_correct = result_handling_report["correct"]
rh_total = result_handling_report["total"]
rh_accuracy = result_handling_report["accuracy"]
print(f"\nResult-handling test accuracy: {rh_correct}/{rh_total} "
      f"= {rh_accuracy*100:.0f}%")
print("\nAll FOUR scenarios used Topic 5's cheap, controllable simulated")
print("dependency -- ZERO real API calls needed to verify this application-")
print("level logic, exactly as the theory recommends for this test category.")
print("\nModule 2 complete. Run Module 3 next.")


MODULE 2: RESULT-HANDLING TESTS (Topic 5's failure categories)

[legitimate_negative] -> PASS
  Actual outcome: status=success, attempts=1

[transient_resolves] -> PASS
  Actual outcome: status=success, attempts=2

[transient_exhausts] -> PASS
  Actual outcome: status=retries_exhausted, attempts=3

[permanent_error] -> PASS
  Actual outcome: status=permanent_failure, attempts=1

Result-handling test accuracy: 4/4 = 100%

All FOUR scenarios used Topic 5's cheap, controllable simulated
dependency -- ZERO real API calls needed to verify this application-
level logic, exactly as the theory recommends for this test category.

Module 2 complete. Run Module 3 next.


### Module 3: Regression Detection — Catching a Schema Change That Breaks Triggering

Save a baseline measurement, simulate a schema/logic change that introduces a regression, and detect it concretely by comparing against the saved baseline.

In [3]:

# ------------------------------------------------------------------
# MODULE 3: Regression detection using a saved baseline
# ------------------------------------------------------------------

# save the CURRENT triggering accuracy as a baseline
baseline_report = suite.run_triggering_tests()
saved_baseline_accuracy = baseline_report["accuracy"]

print("=" * 70)
print("MODULE 3: REGRESSION DETECTION")
print("=" * 70)
print(f"\nSAVED BASELINE triggering accuracy: {saved_baseline_accuracy*100:.0f}%")

# simulate a REGRESSION: someone "improves" the triggering logic in a
# way that accidentally breaks a case that previously worked
def simulate_agent_tool_selection_REGRESSED(email_text: str) -> dict:
    """A deliberately-introduced regression -- the reference-number
    pattern was accidentally narrowed to require lowercase 'fd' in the
    prefix check, breaking real uppercase reference numbers."""
    match = re.search(r'\b[A-Z]{2}\d{4}fd\d{4}\b', email_text)  # BUG: lowercase 'fd' won't match real refs
    if match:
        return {"tool": "validate_fd_reference", "arguments": {"reference_number": match.group(0)}}
    return {"tool": None, "arguments": None}


def run_triggering_tests_with_function(test_cases: list, selection_func) -> dict:
    correct = 0
    for email, correct_tool in test_cases:
        decision = selection_func(email)
        if decision["tool"] == correct_tool:
            correct += 1
    return {"correct": correct, "total": len(test_cases), "accuracy": correct / len(test_cases)}


new_report = run_triggering_tests_with_function(suite.triggering_cases, simulate_agent_tool_selection_REGRESSED)
new_accuracy = new_report["accuracy"]

print(f"NEW (post-change) triggering accuracy: {new_accuracy*100:.0f}%")

regression_detected = new_accuracy < saved_baseline_accuracy
print(f"\nRegression detected (new accuracy < saved baseline)? {regression_detected}")

if regression_detected:
    drop = (saved_baseline_accuracy - new_accuracy) * 100
    print(f"Accuracy DROPPED by {drop:.0f} percentage points -- this change should")
    print(f"be BLOCKED from deployment until investigated, exactly per the")
    print(f"regression-testing discipline this topic describes. Without this")
    print(f"saved baseline to compare against, this regression could have")
    print(f"shipped completely undetected.")

print("\nModule 3 complete. All Chapter 10 Topic 7 modules done.")
print()
print("=" * 70)
print("CHAPTER 10 TOPIC 7 -- KEY POINTS TO REMEMBER")
print("=" * 70)
print("""
  THREE distinct test categories, each catching different failures:
  triggering/selection, argument-correctness, and result-handling --
  demonstrated as REAL, structured, reusable test infrastructure, not
  one-off illustrative measurements.

  Result-handling tests should use CHEAP, CONTROLLABLE simulated
  failures (Topic 5's infrastructure) -- ZERO real API calls needed,
  since this tests application logic, not model behavior.

  Regression testing REQUIRES a saved baseline -- demonstrated
  concretely: a deliberately-introduced regression was caught ONLY
  because a baseline existed to compare against.

  This pre-deployment test suite COMPLEMENTS, never replaces, ongoing
  production monitoring (Chapter 9 Topic 7) -- tests catch known,
  anticipated patterns; monitoring catches genuinely novel ones.

  Every illustrative test set from this chapter's earlier topics
  (Topics 1, 4, 6) is the STARTING MATERIAL for this exact kind of
  real, ongoing test suite -- not disposable, one-time demonstrations.
""")


MODULE 3: REGRESSION DETECTION

SAVED BASELINE triggering accuracy: 100%
NEW (post-change) triggering accuracy: 50%

Regression detected (new accuracy < saved baseline)? True
Accuracy DROPPED by 50 percentage points -- this change should
be BLOCKED from deployment until investigated, exactly per the
regression-testing discipline this topic describes. Without this
saved baseline to compare against, this regression could have
shipped completely undetected.

Module 3 complete. All Chapter 10 Topic 7 modules done.

CHAPTER 10 TOPIC 7 -- KEY POINTS TO REMEMBER

  THREE distinct test categories, each catching different failures:
  triggering/selection, argument-correctness, and result-handling --
  demonstrated as REAL, structured, reusable test infrastructure, not
  one-off illustrative measurements.

  Result-handling tests should use CHEAP, CONTROLLABLE simulated
  failures (Topic 5's infrastructure) -- ZERO real API calls needed,
  since this tests application logic, not model behavior.
